In [ ]:
# import libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
import zarr
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

In [ ]:
# setting path and listing data samples

DATASET = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
TRAIN = DATASET / "train"

print(TRAIN)

zarr_files = sorted(TRAIN.glob("*.zarr"))
geff_files = sorted(TRAIN.glob("*.geff"))

print(f"{len(zarr_files)} image volumes")
print(f"{len(geff_files)} annotation files")

print("\nFirst few samples:\n")

for f in zarr_files[:5]:
    print(f.name)

In [ ]:
# Inspecting a single sample from zarr files

sample = zarr_files[0]

store_sample = zarr.open(sample, mode="r")

images = store_sample["0"]

print("Shape :", images.shape)
print("dtype :", images.dtype)

# looking at one timepoint

t = 0

volume = images[t]

print(volume.shape) # shape = {t, z, y, x}

# checking middle slice

z = volume.shape[0] // 2

plt.figure(figsize=(7,7))
plt.imshow(volume[z], cmap="gray")
plt.title(f"Time {t}, Z {z}")
plt.axis("off")
plt.show()

#viewing several slices
fig, axes = plt.subplots(2,4, figsize=(12,6))

z_slice = [0,8,16,24,32,40,48,63]

for ax, z in zip(axes.flat, z_slice):
    ax.imshow(volume[z], cmap="gray")
    ax.set_title(f"Z={z}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Animating z slices over time

z = images.shape[1] // 2

fig, ax = plt.subplots(figsize=(6,6))

im = ax.imshow(images[0, z], cmap="gray")
ax.axis("off")

def update(frame):
    im.set_array(images[frame, z])
    ax.set_title(f"Time = {frame}")
    return [im]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=10, #images.shape[0] for entire depth
    interval=120,
)

HTML(ani.to_jshtml())

In [ ]:
# Inspecting geff files - single sample
sample_name = sample.stem

geff = zarr.open(TRAIN / f"{sample_name}.geff", mode="r")

print(geff.tree())